# 08 — Bases de données & stockage

**Objectif :** comprendre **où vivent les données** d'une appli, **quelles options** existent, et **comment Azure** offre tout ça en managé.

À l'issue, tu sauras :
- ce qu'est une **base de données**, la différence **SQL vs NoSQL**, **quand choisir quoi** ;
- ce qu'est un **stockage objet** (fichiers/blobs) et pourquoi c'est différent d'une DB ;
- les **services Azure** : Azure SQL, PostgreSQL Flexible Server, **Cosmos DB**, **Storage Account** (Blob/Queue/Table/File), Redis ;
- pourquoi `mailroom-ia` utilise **Cosmos + Blob + Queue** ensemble.

> 🔜 Le **notebook 09** enchaîne sur **Bicep / Infrastructure as Code** pour déployer tout ça de manière reproductible.


## 1. Pourquoi a-t-on besoin d'une base de données ?

Quand ton appli s'arrête, **tout ce qui était en mémoire (RAM) est perdu**. Si tu veux qu'un utilisateur retrouve ses données la prochaine fois qu'il se connecte, il faut les **persister** quelque part.

**Les 3 options pour persister des données** :

| Option | Quand l'utiliser | Limite |
|--------|------------------|--------|
| **Fichier sur disque** | Config, logs, mini POC perso | Pas de requêtes, pas de concurrence, pas de scaling |
| **Base de données** (SQL ou NoSQL) | Données structurées, requêtes, transactions, multi-utilisateurs | Coût, latence réseau |
| **Stockage objet** (Blob, S3) | Fichiers (PDFs, images, vidéos, backups) — gros volumes | Pas de requêtes structurées |

**En vrai, une appli moderne utilise les trois en combinaison.** Notre projet `mailroom-ia` aussi :
- **Cosmos DB** : métadonnées des documents (id, classification, client, statut…).
- **Blob Storage** : les fichiers PDFs eux-mêmes.
- **Queue Storage** : pour orchestrer le pipeline async.


## 2. SQL vs NoSQL — le grand débat

Il y a **deux grandes familles** de bases de données. Choisir la mauvaise pour ton besoin = des mois de galère.

### 2.1 SQL (bases relationnelles)

**SQL** = **S**tructured **Q**uery **L**anguage. Inventé dans les années 70 (IBM, puis Oracle).

Les données sont organisées en **tables** avec des **colonnes typées** et un **schéma rigide** :

```
TABLE clients
┌────┬─────────────┬─────────────────────┬─────────────────────┐
│ id │ displayName │ email               │ createdAt           │
├────┼─────────────┼─────────────────────┼─────────────────────┤
│ 1  │ Jean Dupont │ jean@dupont.fr      │ 2026-01-15 09:30:00 │
│ 2  │ Marie Curie │ marie@curie.fr      │ 2026-02-20 14:15:00 │
└────┴─────────────┴─────────────────────┴─────────────────────┘

TABLE documents
┌────┬───────────┬──────────────┬───────────┐
│ id │ clientId  │ originalName │ category  │
├────┼───────────┼──────────────┼───────────┤
│ A  │ 1         │ facture.pdf  │ factures  │
│ B  │ 1         │ contrat.pdf  │ contrats  │
│ C  │ 2         │ courrier.pdf │ courriers │
└────┴───────────┴──────────────┴───────────┘
```

Les tables sont **liées entre elles** par des **foreign keys** (`documents.clientId` → `clients.id`).

#### Le langage SQL — 5 requêtes à connaître

```sql
-- Lire
SELECT * FROM clients WHERE email = 'jean@dupont.fr';

-- Jointure (puissance #1 de SQL)
SELECT c.displayName, d.originalName, d.category
FROM clients c
JOIN documents d ON d.clientId = c.id
WHERE c.displayName = 'Jean Dupont';

-- Insérer
INSERT INTO clients (id, displayName, email) VALUES (3, 'Paul Verlaine', 'paul@verlaine.fr');

-- Modifier
UPDATE clients SET email = 'jean.dupont@nouveau.fr' WHERE id = 1;

-- Supprimer
DELETE FROM documents WHERE id = 'A';
```

#### Les forces de SQL

| Force | Pourquoi c'est puissant |
|-------|-------------------------|
| **Schéma strict** | Tu ne peux pas écrire `email = 42` (typage). Garanti par la DB, pas par ton code. |
| **Jointures** | "Donne-moi tous les docs des clients de Paris" = 1 requête, calculée par la DB |
| **Transactions ACID** | "Crédite A de 100€ ET débite B de 100€" — soit les 2, soit aucun. Crucial pour la finance. |
| **Standards** | SQL est portable entre PostgreSQL, MySQL, Oracle, SQL Server, SQLite (à 90%). |
| **Maturité** | 50 ans d'optimisations, index, tuning. Énorme communauté. |

#### Les faiblesses de SQL

| Faiblesse | Concrètement |
|-----------|--------------|
| **Schéma rigide** | Ajouter une colonne sur une table de 100M lignes = migration risquée. |
| **Scaling horizontal complexe** | Distribuer une DB SQL sur 10 serveurs = sharding manuel pénible. |
| **Mal adapté aux données hiérarchiques** | Stocker un JSON profond = tables imbriquées + jointures sans fin. |

#### Les SQL connues

- **PostgreSQL** ⭐ — open source, le plus moderne (JSON, GIS, full-text). Le défaut sain.
- **MySQL / MariaDB** — historique LAMP, populaire pour les blogs/CMS.
- **SQL Server** — Microsoft, lourd, écosystème entreprise.
- **Oracle** — entreprise, cher, robuste.
- **SQLite** — fichier local, parfait pour mobile/desktop/embedded.

### 2.2 NoSQL (bases non-relationnelles)

**NoSQL** = "**Not Only SQL**". Famille hétérogène, popularisée à partir de 2009 (Cassandra, MongoDB, Redis). On dit "NoSQL" mais c'est en fait **4 sous-familles très différentes** :

| Sous-famille | Exemple | Pour quoi |
|--------------|---------|-----------|
| **Document** | MongoDB, **Cosmos DB**, CouchDB | Stocker des JSON, schéma souple |
| **Clé-Valeur** | Redis, DynamoDB, etcd | Cache, session, lookup ultra-rapide |
| **Colonne large** | Cassandra, HBase, Bigtable | Énormes volumes time-series |
| **Graphe** | Neo4j, Cosmos DB (Gremlin) | Relations complexes (réseaux sociaux, fraude) |

#### Exemple — DB Document (MongoDB / Cosmos DB)

Au lieu de tables, on stocke des **documents JSON** :

```json
{
  "id": "doc-A",
  "clientId": "1",
  "originalName": "facture.pdf",
  "category": "factures",
  "classification": {
    "model": "gpt-5-mini",
    "confidence": 0.92,
    "reasoning": "Logo EDF en haut, montant total identifié",
    "tags": ["énergie", "EDF", "2026"]
  },
  "history": [
    { "at": "2026-06-08T10:00:00Z", "event": "uploaded" },
    { "at": "2026-06-08T10:00:25Z", "event": "classified" }
  ]
}
```

Pas besoin d'ajouter une colonne — tu rajoutes simplement un champ dans le JSON. Pas besoin de tables séparées pour l'historique — il vit dans le doc.

#### Les forces de NoSQL

| Force | Concrètement |
|-------|--------------|
| **Schéma souple** | Tu changes la forme de tes données en codant, sans migration |
| **Scaling horizontal natif** | Conçu pour être distribué sur 100+ serveurs sans réécrire le code |
| **Performance prévisible** | Cosmos garantit la latence (< 10 ms) car indexé automatiquement |
| **Naturel pour les apps modernes** | Ton code manipule du JSON, ta DB stocke du JSON, zéro mismatch |

#### Les faiblesses de NoSQL

| Faiblesse | Concrètement |
|-----------|--------------|
| **Pas de jointures** | "Tous les docs des clients de Paris" = 2 requêtes + jointure côté code |
| **Pas de transactions multi-doc** (souvent) | Plus dur à garantir la cohérence sur opérations complexes |
| **Pas de validation native** | C'est à ton code d'imposer la structure (Zod, Pydantic…) |
| **Coût** | Cosmos peut vite coûter cher si mal partitionné |

### 2.3 Comment choisir ?

| Critère | SQL | NoSQL |
|---------|-----|-------|
| Données fortement liées (banque, ERP, RH) | ✅ | ❌ |
| Schéma stable, contraintes strictes | ✅ | ❌ |
| Transactions complexes (paiement) | ✅ | ⚠️ |
| Schéma qui évolue souvent | ⚠️ | ✅ |
| Document JSON / objet métier complet en une entrée | ⚠️ | ✅ |
| Scaling horizontal massif (millions d'utilisateurs) | ⚠️ | ✅ |
| Latence garantie (< 10 ms) | ⚠️ | ✅ |
| Recherche full-text rapide | ⚠️ | ⚠️ → en vrai **Azure AI Search** |
| Données hiérarchiques (commentaires, historique) | ⚠️ | ✅ |

**Règle pragmatique** :
- En doute → commence par **PostgreSQL**. C'est le défaut sain pour 80% des apps.
- Données = JSON document naturel et tu veux la perf garantie d'Azure → **Cosmos DB**.
- Cache / session / lookup ultra rapide → **Redis** en complément.

**Tu peux (et tu dois souvent) combiner les deux.** Ex. un e-commerce : PostgreSQL pour commandes/paiement, Cosmos pour le catalogue produits, Redis pour les sessions.


## 3. Les bases Azure managées

Azure propose plusieurs services DB **managés** = Microsoft s'occupe de l'OS, des patchs, des backups, du HA. Tu manipules juste tes données.

### 3.1 Carte d'identité des services DB Azure

| Service | Type | Quand l'utiliser |
|---------|------|------------------|
| **Azure SQL Database** | SQL (basé sur SQL Server) | Apps .NET, équipes déjà sur SQL Server |
| **Azure Database for PostgreSQL Flexible Server** ⭐ | SQL (PostgreSQL) | Le défaut sain pour apps modernes Python/Node |
| **Azure Database for MySQL Flexible Server** | SQL (MySQL) | App PHP/WordPress, migration depuis MySQL |
| **Azure Cosmos DB** ⭐ | NoSQL multi-modèle (Document, Graph, Cassandra, Mongo, Table) | App scalable mondiale, document JSON, **notre choix mailroom-ia** |
| **Azure Cache for Redis** | NoSQL Clé-Valeur | Cache, session, rate-limit, pub/sub léger |
| **Azure Table Storage** | NoSQL Clé-Valeur basique | Données simples très haute volume, ultra peu cher |
| **Azure AI Search** | Index recherche (Lucene) | Full-text + vector search (RAG, recherche produit) |

### 3.2 Azure SQL Database (le SQL Microsoft)

C'est SQL Server, managé. Tu n'installes rien, tu cliques sur "create".

- **3 modèles de tarif** : DTU (simple), vCore (flexible), Serverless (pause-able).
- **Backups auto** (PITR jusqu'à 35 jours).
- **Haute dispo** intégrée (option Business Critical → 99.99 %).
- **Géo-réplication** : réplique vers une autre région pour DR.
- **Authentification** : Entra ID + SQL natif (préfère Entra).

Quand le choisir : équipe déjà sur la stack Microsoft, besoin de SQL Server features (T-SQL avancé, In-Memory OLTP).

### 3.3 PostgreSQL Flexible Server

PostgreSQL managé par Azure. Le défaut pour app Python/Node.

- **PostgreSQL upstream** (pas un fork), versions 14/15/16/17.
- **Tarif** : Burstable (B1ms = ~12 €/mois, parfait POC) → General Purpose → Memory Optimized.
- **Backups auto**, point-in-time restore.
- **Extensions** : `pgvector` pour stocker des embeddings IA, `PostGIS` pour le géospatial.
- **Authentification** : Entra ID (passwordless 👍) ou password classique.

Quand le choisir : app moderne, équipe à l'aise avec PostgreSQL, besoin de SQL standard.

### 3.4 Cosmos DB — la NoSQL globale d'Azure

C'est une **base NoSQL distribuée multi-modèle**. Une seule techno qui te donne le choix de l'API :

| API Cosmos | Tu écris du… |
|-----------|--------------|
| **NoSQL (SQL)** ⭐ | SQL-like sur JSON (notre choix) |
| **MongoDB** | Mongo natif (migration depuis Mongo) |
| **Cassandra** | CQL |
| **Gremlin** | Graphes |
| **Table** | Compatible Azure Table Storage |

#### Concepts Cosmos à connaître

- **Account** = la ressource Azure facturée (ex. `cosmos-stage-sadk`).
- **Database** = groupe logique de containers (ex. `mailroom`).
- **Container** = "table" Cosmos (ex. `documents`, `clients`). ⚠️ **À ne pas confondre avec Docker container !**
- **Item** = un document JSON. A obligatoirement un champ `id` unique dans sa partition.
- **Partition key** = ⚡ **LA décision la plus importante**. Détermine sur quelle machine physique vit ton item. Bonne PK = perf au top, mauvaise PK = "hot partition" qui rame.
- **Request Unit (RU)** = l'unité de coût. 1 lecture par id ≈ 1 RU, 1 écriture ≈ 5 RU, 1 query cross-partition = 10-50 RU.

#### Choisir la partition key

| Données | Bonne PK | Pourquoi |
|---------|----------|----------|
| Commandes par client | `/clientId` | Les commandes d'un client sont lues/écrites ensemble |
| Logs par jour | `/date` (ex: `"2026-06-08"`) | Time-series naturel, mais attention au "hot date" si toujours le même |
| Documents par tenant | `/tenantId` | Multi-tenant SaaS |
| Sessions utilisateurs | `/userId` | Une session = un user |

**Notre `mailroom-ia`** : container `documents` partitionné par `/clientId` (les docs d'un client sont colocalisés → query par client = rapide).

#### Deux modes de facturation

| Mode | Quand | Coût |
|------|-------|------|
| **Serverless** ⭐ | POC, trafic faible/imprévisible | Tu payes par RU consommée. Quelques cents/mois pour 100 docs. |
| **Provisioned throughput** | Prod à charge stable | Tu réserves N RU/s 24/7. Plus cher mais latence garantie. |

#### RBAC Cosmos data-plane (le piège classique)

Azure RBAC (`Cosmos DB Account Reader`, `Cosmos DB Contributor`…) donne accès au **management plane** : créer un compte, lister les databases, voir les métriques.

**Il NE donne PAS le droit de lire ou écrire des items !**

Pour ça, il faut un **rôle Cosmos natif** assigné via `sqlRoleAssignments`. Le rôle built-in `Cosmos DB Built-in Data Contributor` (id `00000000-0000-0000-0000-000000000002`) donne CRUD complet.

C'est ce qu'on fait pour les UAMI `web` et `worker` dans `mailroom-ia` (cf. setup.ipynb §7.5).

### 3.5 Redis — le cache mémoire ultra-rapide

**Azure Cache for Redis** = Redis managé. Tu l'utilises pour :

- **Cache** : tu stockes la réponse d'une query Cosmos pendant 60 s pour pas la refaire à chaque hit.
- **Session** : `sessionId → userJSON` (latence 1 ms).
- **Rate-limiting** : `userId → counter` (compteur atomique).
- **Pub/Sub** léger.
- **Distributed lock** : "1 seul worker peut traiter ce job à la fois".

Tarifs : Basic (POC), Standard (HA), Premium (cluster + persistence).

> 💡 Redis stocke en RAM → **ultra rapide** mais **volatile par défaut**. Pour les données critiques, garde ta source de vérité dans une vraie DB et utilise Redis comme cache.


## 4. Stockage objet — Blob Storage

Une base de données est conçue pour des **petites entrées structurées** (un user, un produit, un doc JSON). **Elle est mauvaise pour stocker des gros fichiers** (PDFs, images, vidéos, backups, ML datasets…).

Pour ça, on utilise un **stockage objet** : un système qui te laisse **uploader/downloader des fichiers** identifiés par un **nom (clé)**.

### 4.1 Concept stockage objet

```
Container "mailroom"
 ├── _inbox/
 │    ├── 7e837744-….pdf       (Hot tier, 250 KB)
 │    └── 8a004307-….pdf       (Hot tier, 1.2 MB)
 ├── clients/
 │    ├── client42/
 │    │    ├── factures/2026/edf-feb.pdf
 │    │    └── contrats/assurance/axa.pdf
 │    └── client99/
 │         └── courriers/medical/ordo.pdf
 └── archives/
      └── 2024/...
```

Le `/` dans les noms est **purement conventionnel** — il n'y a pas de vrais dossiers, c'est juste un préfixe dans le nom du blob. L'UI Azure Portal et les SDK affichent une "arborescence" pour l'UX, mais physiquement tout est plat.

### 4.2 Azure Storage Account — les 4 services

Un Storage Account Azure expose **4 services** derrière le même endpoint :

| Service | Endpoint | Pour quoi |
|---------|----------|-----------|
| **Blob** ⭐ | `<acct>.blob.core.windows.net` | Fichiers jusqu'à 5 TB. Notre choix pour les PDFs. |
| **Queue** ⭐ | `<acct>.queue.core.windows.net` | File d'attente FIFO simple, 64 KB max/message |
| **Table** | `<acct>.table.core.windows.net` | NoSQL clé/valeur basique (pré-Cosmos) |
| **File** | `<acct>.file.core.windows.net` | Partage SMB/NFS (= un disque réseau) |

### 4.3 Blob Storage — les concepts

- **Storage Account** = le compte (`stmailroomsadk`).
- **Container** = le "bucket" (le nôtre s'appelle `mailroom`). ⚠️ **Encore un mot surchargé** — ce n'est ni Docker ni Cosmos.
- **Blob** = le fichier. Identifié par son nom (peut contenir des `/`).
- **Block Blob** = type le plus courant (fichiers immutables).
- **Append Blob** = optimisé pour append (logs).
- **Page Blob** = pour les VHD (disques de VM).

### 4.4 Les tiers de stockage — clé pour optimiser les coûts

Pas tous les fichiers ont besoin d'être accessibles en 10 ms. Azure propose **4 tiers** :

| Tier | Latence d'accès | Stockage (€/Go/mois) | Retrieval (€/Go) | Quand |
|------|----------------|----------------------|------------------|-------|
| **Hot** | Immédiate | ~0.02 € | Gratuit | Fichiers actifs (notre `_inbox/`) |
| **Cool** | Immédiate | ~0.01 € | ~0.01 € | Backup mensuel, archives récentes |
| **Cold** | Immédiate | ~0.005 € | ~0.03 € | Archives long terme |
| **Archive** | **~15h** (rehydrate) | ~0.001 € | ~0.05 € | Compliance, "on garde 7 ans au cas où" |

Tu peux automatiser avec une **Lifecycle Policy** : "déplace les blobs > 30 jours en Cool, > 1 an en Archive".

### 4.5 Sécurité — accès aux blobs

| Méthode | Description | Quand |
|---------|-------------|-------|
| **Access keys** | 2 clés racines, accès total | À éviter en prod, OK pour POC vite fait |
| **SAS token** (Shared Access Signature) | Token signé qui donne accès limité (1 blob, lecture seule, expire dans 1h) | Donner accès à un client externe sans clé permanente |
| **Entra ID + RBAC** ⭐ | Identité Entra → token → autorisation par rôle Azure | **Notre choix** : Managed Identity du worker pose un PDF dans `_inbox/` |
| **Anonymous public** | Lecture publique sans auth | Sites statiques, images publiques |

**Rôles RBAC Blob importants** :
- `Storage Blob Data Contributor` : lecture/écriture (notre web + worker).
- `Storage Blob Data Reader` : lecture seule.
- `Storage Blob Data Owner` : tout + gestion ACL POSIX.

### 4.6 Queue Storage — le pipeline asynchrone

Une **queue FIFO** ultra simple, idéale pour découpler producteur (web) et consommateur (worker).

```
[web]  →  PUT message {id: "doc-A", blobName: "..."}  →  [Queue: doc-to-classify]
                                                                │
                                                                ▼
                                              [KEDA poll toutes les 30 s]
                                                                │
                                                                ▼
                                                  [worker lance 1 execution]
                                                                │
                                                                ▼
                                              GET message + DELETE quand done
```

**Concepts** :
- **Visibility timeout** : quand le worker `get` un message, il devient invisible pour les autres pendant N secondes. S'il ne le `delete` pas avant la fin → il redevient visible (retry automatique).
- **Dequeue count** : combien de fois ce message a été lu sans delete. Au-delà d'un seuil → "poison message" à investiguer.
- **Max 64 KB** par message → tu y mets une **référence** (id, nom de blob), pas le contenu.

### 4.7 Comparaison messaging Azure (pour info)

| Service | Best for | Notre projet |
|---------|----------|--------------|
| **Storage Queue** ⭐ | POC, volume modéré, ultra simple | ✅ |
| **Service Bus** | Entreprise — sessions, dead-letter, transactions | — |
| **Event Hubs** | Stream haut débit (millions d'evts/s) | — |
| **Event Grid** | Pub/sub d'événements Azure ("blob créé" → handler) | — |

Plus de détail dans le notebook setup `mailroom-ia` §0.B.


## 5. Notre projet `mailroom-ia` — l'orchestration complète

Récap de ce qu'on utilise et pourquoi.

```
   [Admin uploade un PDF]
            │
            ▼
   ┌─────────────────────┐
   │ web (Next.js)       │
   │ - PUT blob dans     │
   │   mailroom/_inbox/  │
   │ - poste message     │
   │   dans Queue        │
   └─────────────────────┘
            │
            │
   ┌────────┼────────────────────────────┐
   ▼        ▼                            ▼
┌──────┐ ┌───────┐                  ┌────────────┐
│ BLOB │ │ QUEUE │                  │ Cosmos DB  │
│      │ │       │                  │ documents/ │
│ PDF  │ │ ref   │                  │ clients/   │
└──────┘ └───────┘                  └────────────┘
            │                            ▲
            │ KEDA scale                 │
            ▼                            │
   ┌─────────────────────┐               │
   │ worker (Python)     │ ──── upsert ──┘
   │ - DI + LLM Foundry  │
   │ - move blob         │
   │ - upsert Cosmos     │
   └─────────────────────┘
```

| Service | Pour quoi | Pourquoi pas un autre ? |
|---------|-----------|-------------------------|
| **Blob Storage** | Stocker les PDFs (jusqu'à 25 MB chacun) | Cosmos a une limite de 2 MB par item, et stocker un PDF en base64 dans une DB c'est lent et cher |
| **Queue Storage** | Découpler web (synchrone) du worker (async, lourd) | Service Bus = overkill pour un POC, Event Grid = pas le bon pattern ici |
| **Cosmos DB** | Métadonnées des docs + clients (JSON souple) | Notre data est JSON natif (classification IA avec champs variables), schéma évolutif → NoSQL gagne. PostgreSQL aurait aussi marché. |

Le **Storage Account** = `stmailroomsadk` (Blob + Queue dans le même compte).
La **Cosmos DB** = `cosmos-stage-sadk` (Database `mailroom`, containers `documents` + `clients`).


## 6. Pour aller plus loin

- **Doc Cosmos DB** : https://learn.microsoft.com/azure/cosmos-db/
- **Doc Storage Account** : https://learn.microsoft.com/azure/storage/
- **Doc PostgreSQL Flexible** : https://learn.microsoft.com/azure/postgresql/flexible-server/
- **Choisir une DB Azure** (decision tree officiel) : https://learn.microsoft.com/azure/architecture/guide/technology-choices/data-store-decision-tree
- **Cosmos DB best practices** : https://learn.microsoft.com/azure/cosmos-db/nosql/best-practice-dotnet
- **Cosmos Capacity Calculator** : https://cosmos.azure.com/capacitycalculator/

### 🚀 Prochaine étape : **notebook 09 — Bicep & Infrastructure as Code**

Tu as découvert tous les services qu'on utilise dans `mailroom-ia` (notebooks 04-08). Maintenant on va apprendre à les **déployer en code** avec Bicep, plutôt qu'avec des commandes `az` impératives.
